# 08 — Final Comparison & Ablation Study
## Comprehensive Model Evaluation and Results Synthesis

This notebook brings together all results into final comparison tables,
generates PR/ROC curves for all models, and produces the ablation study.

Produces: V24 (confusion matrices), V25 (PR curves), V26 (ROC curves),
          V29 (final comparison table), V30 (ablation table), V39 (ablation bar chart)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

from src.evaluation.metrics import (
    full_evaluation, plot_precision_recall_curves, plot_roc_curves
)
from src.evaluation.comparison import (
    build_comparison_table, build_ablation_table, plot_ablation_bar_chart
)

FIGURES_DIR = Path('../outputs/figures')
TABLES_DIR = Path('../outputs/tables')
MODELS_DIR = Path('../outputs/models')
PROCESSED_DIR = Path('../data/processed')

for d in [FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Load All Models and Test Data

In [ ]:
from sklearn.preprocessing import LabelEncoder

X_test = pd.read_csv(PROCESSED_DIR / 'baf_X_test.csv')
y_test = pd.read_csv(PROCESSED_DIR / 'baf_y_test.csv').squeeze()

print(f'Test set: {X_test.shape[0]:,} samples, {y_test.mean()*100:.2f}% fraud')

# Load all available models — only keep those matching BAF feature count
models = {}
model_files = {
    'Logistic Regression': 'logistic_regression.joblib',
    'Random Forest': 'random_forest.joblib',
    'XGBoost': 'xgboost.joblib',
    'XGB-RF Ensemble (Centralized)': 'xgb_rf_ensemble.joblib',
    'XGB-RF Ensemble (GA-Optimized)': 'ga_optimized_ensemble.joblib',
}

for name, filename in model_files.items():
    try:
        m = joblib.load(MODELS_DIR / filename)
        nf = getattr(m, 'n_features_in_', X_test.shape[1])
        if nf != X_test.shape[1]:
            print(f'  Skipped: {name} ({nf} features != {X_test.shape[1]})')
            continue
        models[name] = m
        print(f'  Loaded: {name}')
    except FileNotFoundError:
        print(f'  Not found: {name}')

# FL Ensemble — use BAF-specific models
try:
    from sklearn.ensemble import VotingClassifier
    fl_xgb = joblib.load(MODELS_DIR / 'fl_baf_xgb.joblib')
    fl_rf = joblib.load(MODELS_DIR / 'fl_baf_rf.joblib')

    n_xgb = getattr(fl_xgb, 'n_features_in_', X_test.shape[1])
    n_rf = getattr(fl_rf, 'n_features_in_', X_test.shape[1])
    if n_xgb == X_test.shape[1] and n_rf == X_test.shape[1]:
        fl_ensemble = VotingClassifier(
            estimators=[('xgb', fl_xgb), ('rf', fl_rf)], voting='soft',
        )
        fl_ensemble.estimators_ = [fl_xgb, fl_rf]
        le = LabelEncoder()
        le.classes_ = np.array([0, 1])
        fl_ensemble.le_ = le
        fl_ensemble.classes_ = np.array([0, 1])
        models['XGB-RF Ensemble (Federated)'] = fl_ensemble
        print(f'  Loaded: XGB-RF Ensemble (Federated) [BAF]')
    else:
        print(f'  FL feature mismatch: XGB={n_xgb}, RF={n_rf}, test={X_test.shape[1]}')
except FileNotFoundError:
    print('  FL BAF models not found — run notebook 05 first')
except Exception as e:
    print(f'  FL Ensemble not available: {e}')

print(f'\nTotal models loaded: {len(models)}')

## 2. V24 — Confusion Matrices + Full Evaluation

In [ ]:
all_metrics = []
all_probas = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = full_evaluation(y_test, y_pred, y_proba, name,
                              output_dir=str(FIGURES_DIR))
    all_metrics.append(metrics)
    all_probas[name] = y_proba
    
    print(f'{name}:')
    print(f'  AUPRC={metrics["AUPRC"]:.4f}  F1={metrics["F1-Score"]:.4f}  '
          f'Recall={metrics["Recall"]:.4f}  ROC-AUC={metrics["ROC-AUC"]:.4f}')

print('\nConfusion matrices saved for all models.')

## 3. V25 — Precision-Recall Curves

In [ ]:
plot_precision_recall_curves(all_probas, y_test, output_dir=str(FIGURES_DIR))
print('Saved: precision_recall_curves_all.png')

## 4. V26 — ROC Curves

In [ ]:
plot_roc_curves(all_probas, y_test, output_dir=str(FIGURES_DIR))
print('Saved: roc_curves_all.png')

## 5. V29 — Final Comparison Table

In [ ]:
# Load latency results if available
try:
    latency_df = pd.read_csv(TABLES_DIR / 'latency_results.csv')
    latency_map = dict(zip(latency_df['Model'], latency_df['mean_ms'].astype(float)))
except (FileNotFoundError, KeyError):
    latency_map = None
    print('Latency results not found — run notebook 07 first.')

comparison_df = build_comparison_table(
    all_metrics, latency_results=latency_map,
    output_dir=str(TABLES_DIR)
)
print('\nSaved: final_comparison.csv')

## 6. V30 + V39 — Ablation Study

In [ ]:
# Build ablation study from collected metrics
ablation_configs = [
    'XGBoost',
    'Random Forest',
    'XGB-RF Ensemble (Centralized)',
    'XGB-RF Ensemble (GA-Optimized)',
    'XGB-RF Ensemble (Federated)',
]

ablation_data = []
for config in ablation_configs:
    matching = [m for m in all_metrics if m['Model'] == config]
    if matching:
        m = matching[0]
        is_fl = 'Federated' in config
        ablation_data.append({
            'Configuration': config,
            'AUPRC': round(m['AUPRC'], 4),
            'F1-Score': round(m['F1-Score'], 4),
            'Recall': round(m['Recall'], 4),
            'Latency (ms)': (latency_map.get(config, 'N/A')
                             if latency_map else 'N/A'),
            'Privacy': 'Yes' if is_fl else 'No',
            'Explainable': 'Yes',  # All use SHAP
        })

if ablation_data:
    ablation_df = build_ablation_table(ablation_data, output_dir=str(TABLES_DIR))
    print('\nSaved: ablation_study.csv')
    
    # V39 — Ablation bar chart
    plot_ablation_bar_chart(ablation_df, output_dir=str(FIGURES_DIR))
    print('Saved: ablation_bar_chart.png')
else:
    print('Not enough models for ablation study.')

## 7. Summary

In [ ]:
print('=== Final Results Summary ===')
print(f'\nModels evaluated: {len(all_metrics)}')
print(f'\nBest model by AUPRC:')
best = max(all_metrics, key=lambda x: x['AUPRC'])
print(f'  {best["Model"]}: AUPRC={best["AUPRC"]:.4f}')

print(f'\nBest model by F1-Score:')
best_f1 = max(all_metrics, key=lambda x: x['F1-Score'])
print(f'  {best_f1["Model"]}: F1={best_f1["F1-Score"]:.4f}')

print(f'\n--- Files Generated ---')
print('Tables: final_comparison.csv, ablation_study.csv')
print('Figures: confusion_matrix_*.png, precision_recall_curves_all.png,')
print('         roc_curves_all.png, ablation_bar_chart.png')